In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib
plt.style.use('ggplot')

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import LabelEncoder

%matplotlib inline
matplotlib.rcParams['figure.figsize'] = (12, 8)

pd.options.mode.chained_assignment = None

##### Загрузка данных

In [ ]:
horse_colic = fetch_ucirepo(id=47)

X = horse_colic.data.features.copy()
y = horse_colic.data.targets.copy()

df = pd.concat([X, y], axis=1)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace('-', '_', regex=False)
    .str.replace(' ', '_', regex=False)
)

target_col = y.columns[0]
target_col = target_col.strip().lower().replace('-', '_').replace(' ', '_')

print(df)

In [ ]:
print(df.shape)
print(df.dtypes)

In [ ]:
df_numeric = df.select_dtypes(include=[np.number])
numeric_cols = df_numeric.columns.values
print(numeric_cols)

In [ ]:
df_non_numeric = df.select_dtypes(exclude=[np.number])
non_numeric_cols = df_non_numeric.columns.values
print(non_numeric_cols)

### Исследование целевой переменной

In [ ]:
plt.figure(figsize=(8, 6))
df[target_col].value_counts().plot(kind='bar')
plt.xlabel('Класс', fontsize=12)
plt.ylabel('Количество', fontsize=12)
plt.title('Распределение целевой переменной')
plt.show()

print(df[target_col].value_counts())

### Отсутствующие данные

In [ ]:
print(df.describe(include='all'))

In [ ]:
df_new = df.replace(['?', '', ' ', 'NA', 'N/A', 'nan', 'NaN'], np.nan)

print('Количество строк:', df_new.shape[0])
print('Пропущенные данные по столбцам:')
print(df_new.isnull().sum())

### Приведение категориальных данных к единому формату

In [ ]:
cat_cols = df_new.select_dtypes(exclude=[np.number]).columns

for col in cat_cols:
    df_new[col] = df_new[col].astype(str).str.strip().str.lower()

for col in df_new.columns:
    print(df_new[col].value_counts(dropna=False))
    print()

### Дубликаты записей

In [ ]:
df_drop = df_new.drop_duplicates()

print('До удаления дубликатов:', df_new.shape)
print('После удаления дубликатов:', df_drop.shape)

df_drop.head()

### Кодирование категориальных признаков

In [ ]:
df_encoded = df_drop.copy()
encoders = {}

for col in df_encoded.columns:
    labelencoder = LabelEncoder()
    df_encoded[col] = labelencoder.fit_transform(df_encoded[col])
    encoders[col] = dict(zip(labelencoder.classes_, labelencoder.transform(labelencoder.classes_)))

print('Словари кодирования:')
for col, mapping in encoders.items():
    print(col, mapping)

df_encoded.head()

## Нетипичные данные (выбросы)

In [ ]:
feature_cols = [col for col in df_encoded.columns if col != target_col]

df_encoded[feature_cols].boxplot()
plt.title('Boxplot закодированных признаков Horse Colic')
plt.xticks(rotation=45)
plt.show()

In [ ]:
df_encoded[feature_cols].describe()

In [ ]:
def remove_outliers_iqr(data, columns):
    df_result = data.copy()

    for column in columns:
        Q1 = df_result[column].quantile(0.25)
        Q3 = df_result[column].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = (df_result[column] < lower_bound) | (df_result[column] > upper_bound)
        print(column)
        print('Нижняя граница:', lower_bound)
        print('Верхняя граница:', upper_bound)
        print('Количество выбросов:', outliers.sum())
        print()

        df_result = df_result[~outliers]

    return df_result

df_clean = remove_outliers_iqr(df_encoded, feature_cols)

print('Данные до удаления выбросов:', df_encoded.shape)
print('Данные после удаления выбросов:', df_clean.shape)

df_clean.head()

In [ ]:
df_drop.to_csv('horse_colic_cleaned.csv', index=False)